In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import IntervalTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

### Task Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


Train the model on the original task.

In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:19<00:00,  6.46s/it, val_loss=0.0369, val_acc=0.991]

Test Results: [(0.0209, 0.993)]


[(0.0209, 0.993)]

Train subsequent tasks with bounds.

In [6]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=300,
    min_acc_limit=0.9,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, context_id=i)

Test Results: [(0.0209, 0.993)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████| 300/300 [00:21<00:00, 14.21it/s, size=92402.73, obj=14.954, min_soft_acc=0.874]


Final bbox:  Obj=14.95,  Size=92402.73,  Min acc hard=0.88,  Min acc soft=0.88
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 15 checkpoints
Checkpoints sizes: ['92327.42', '92344.30', '92356.60', '92363.93', '92368.88', '92373.45', '92377.42', '92381.19', '92385.02', '92388.44', '92391.20', '92393.94', '92397.19', '92400.30', '92402.73']
Checkpoint certificates: ['0.95', '0.89', '0.89', '0.89', '0.89', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88', '0.88']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 5/5 [00:31<00:00,  6.30s/it, val_loss=0.1170, val_acc=0.9672, proj=0]


Test Results: [(0.0207, 0.9935), (0.099, 0.9708)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0551 (Positive = violated)
Computing Rashomon set within outer box of size: 92327.42
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.95,  Min acc soft=0.96


...

Final bbox:  Obj=7.48,  Size=46218.79,  Min acc hard=0.90,  Min acc soft=0.90
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 15 checkpoints
Checkpoints sizes: ['46180.04', '46191.35', '46201.24', '46207.09', '46209.32', '46211.30', '46212.51', '46213.80', '46215.39', '46216.01', '46217.02', '46217.84', '46218.09', '46218.29', '46218.79']
Checkpoint certificates: ['0.93', '0.93', '0.90', '0.90', '0.89', '0.89', '0.90', '0.90', '0.89', '0.89', '0.89', '0.89', '0.89', '0.90', '0.90']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 5/5 [00:30<00:00,  6.09s/it, val_loss=0.0894, val_acc=0.9705, proj=0]


Test Results: [(0.0207, 0.993), (0.099, 0.9708), (0.0221, 0.9935), (0.0619, 0.9797)]


Training Epochs: 100%|███████████████████████████████| 5/5 [00:33<00:00,  6.73s/it, val_loss=0.0575, val_acc=0.9824, proj=0]


Test Results: [(0.0207, 0.9935), (0.099, 0.9708), (0.0221, 0.9935), (0.0619, 0.9802), (0.0637, 0.9898)]


### Domain Incremental Learning

In [12]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [13]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [14]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:18<00:00,  6.29s/it, val_loss=0.0352, val_acc=0.991]


Test Results: [(0.0208, 0.994)]


[(0.0208, 0.994)]

In [17]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.9,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    batch_size=400,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, epochs=3)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test, domain_map_fn=domain_map_fn)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.90
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|████████████████████████████████████████| 200/200 [00:10<00:00, 19.95it/s, size=1235.65, obj=0.199, min_soft_acc=0.838]


Final bbox:  Obj=0.20,  Size=1235.65,  Min acc hard=0.89,  Min acc soft=0.89
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['30.23', '157.65', '369.99', '541.46', '671.33', '804.54', '919.39', '1035.99', '1140.97', '1235.65']
Checkpoint certificates: ['0.96', '0.95', '0.92', '0.93', '0.91', '0.89', '0.89', '0.89', '0.89', '0.89']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:20<00:00,  6.88s/it, val_loss=3.0543, val_acc=0.1715, proj=0]


Test Results: [(0.0233, 0.994), (3.1826, 0.1679)]
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.0910 (Positive = violated)
Computing Rashomon set within outer box of size: 30.23
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.12
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.22,  Min acc soft=0.21


...

Final bbox:  Obj=0.00,  Size=27.19,  Min acc hard=0.78,  Min acc soft=0.77
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['12.14', '27.12', '27.18', '26.99', '27.08', '27.23', '27.10', '27.09', '27.17', '27.19']
Checkpoint certificates: ['0.77', '0.78', '0.78', '0.78', '0.78', '0.77', '0.78', '0.78', '0.78', '0.78']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:17<00:00,  5.75s/it, val_loss=3.9314, val_acc=0.3882, proj=5]


Test Results: [(0.0249, 0.9935), (3.2326, 0.176), (0.485, 0.8097), (3.6927, 0.4033)]


Training Epochs: 100%|███████████████████████████████| 3/3 [00:23<00:00,  7.98s/it, val_loss=2.6381, val_acc=0.5867, proj=5]


Test Results: [(0.0249, 0.9935), (3.2327, 0.176), (0.4849, 0.8102), (3.6929, 0.4033), (3.0807, 0.5644)]


### Class Incremental Learning

In [18]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [19]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs: 100%|████████████████████████████████████████| 3/3 [00:18<00:00,  6.28s/it, val_loss=0.0363, val_acc=0.991]


Test Results: [(0.0205, 0.9935)]


[(0.0205, 0.9935)]

In [21]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    n_certificate_samples=400,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, epochs=3, batch_size=512)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1850 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.98,  Min acc soft=0.99


100%|████████████████████████████████████████| 200/200 [00:10<00:00, 19.99it/s, size=1349.73, obj=0.217, min_soft_acc=0.781]


Final bbox:  Obj=0.22,  Size=1349.73,  Min acc hard=0.80,  Min acc soft=0.80
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['32.84', '180.06', '391.55', '570.85', '707.13', '841.47', '970.79', '1097.81', '1222.81', '1349.73']
Checkpoint certificates: ['0.88', '0.92', '0.84', '0.81', '0.81', '0.80', '0.79', '0.80', '0.81', '0.80']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:09<00:00,  3.29s/it, val_loss=8.5777, val_acc=0.0000, proj=2]


Test Results: [(0.0219, 0.9935), (9.0355, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


...

Test Results: [(0.0221, 0.9935), (18.1218, 0.0), (7.802, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|███████████████████████████████| 3/3 [00:11<00:00,  3.88s/it, val_loss=9.2780, val_acc=0.0000, proj=0]


Test Results: [(0.0217, 0.9935), (28.6059, 0.0), (15.9719, 0.0), (9.4205, 0.0)]


Training Epochs: 100%|███████████████████████████████| 3/3 [00:11<00:00,  3.81s/it, val_loss=7.5342, val_acc=0.0000, proj=0]


Test Results: [(0.022, 0.9935), (41.316, 0.0), (27.2419, 0.0), (21.5627, 0.0), (7.7613, 0.0)]


### Class Incremental Learning + Regulariser

In [22]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [23]:
unbias = UnbiasRegulariser(lmbd=0.01)
l2 = L2Regulariser(lmbd=0.01)
regulariser = MultiRegulariser([l2, unbias])

trainer = SimpleTrainer(model)
trainer.train(
    train_tasks[0], val_tasks[0], epochs=3, batch_size=256, regulariser=regulariser
)
trainer.test(test_tasks[0:1], regulariser=regulariser)

Training Epochs: 100%|██████████████████████████████████████████| 3/3 [00:24<00:00,  8.17s/it, val_loss=2.29, val_acc=0.991]

Test Results: [(0.0198, 0.9935)]


[(0.0198, 0.9935)]

In [25]:
interval_trainer = IntervalTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.33,
    dual_learning_rate=0.01,
    n_certificate_samples=400,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(
        train, val, batch_size=512, epochs=3, regulariser=regulariser
    )
    interval_trainer.test(test_tasks[0 : i + 1], regulariser=regulariser)
    if i < len(train_tasks) - 2:
        interval_trainer.compute_rashomon_set(test)

---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1850 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.98,  Min acc soft=0.99


100%|████████████████████████████████████████| 200/200 [00:10<00:00, 18.91it/s, size=1309.84, obj=0.211, min_soft_acc=0.790]


Final bbox:  Obj=0.21,  Size=1309.84,  Min acc hard=0.81,  Min acc soft=0.81
Computing final certificates over 400 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['33.02', '182.46', '389.65', '546.79', '679.85', '811.48', '934.85', '1061.18', '1186.58', '1309.84']
Checkpoint certificates: ['0.86', '0.85', '0.84', '0.81', '0.82', '0.80', '0.82', '0.81', '0.80', '0.81']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs: 100%|██████████████████████████████| 3/3 [00:12<00:00,  4.16s/it, val_loss=25.5478, val_acc=0.0000, proj=1]


Test Results: [(0.0296, 0.9905), (22.1493, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


...

Test Results: [(0.0296, 0.9905), (23.0833, 0.0), (19.8048, 0.0)]
---------------------------- Computing Rashomon set ----------------------------


Training Epochs: 100%|██████████████████████████████| 3/3 [00:11<00:00,  3.70s/it, val_loss=26.1814, val_acc=0.0000, proj=0]


Test Results: [(0.0296, 0.9905), (23.0641, 0.0), (20.7175, 0.0), (22.3817, 0.0)]


Training Epochs: 100%|██████████████████████████████| 3/3 [00:14<00:00,  4.91s/it, val_loss=21.9788, val_acc=0.0000, proj=0]


Test Results: [(0.0296, 0.9905), (23.2921, 0.0), (20.9184, 0.0), (23.6272, 0.0), (17.877, 0.0)]
